# Python 基础语法、文件与异常

本 notebook 对应 Junior DE checklist 的 Python 部分：基础语法、文件与异常。目标不是背语法，而是能写出一段小型数据清洗脚本，能读文件、处理 list/dict、写 JSON/CSV，并在报错时定位问题。

| 模块 | 必须掌握 |
|---|---|
| 基础语法 | 变量、数据类型、`if` / `for` / `while` |
| 函数 | 参数、返回值、默认参数、拆分重复逻辑 |
| 容器 | `list` / `dict` / `set` 常用操作 |
| 推导式 | 列表推导式、字典推导式 |
| 字符串 | f-string 格式化 |
| 文件 | `pathlib`、`with open`、JSON、CSV |
| 异常 | 捕获具体异常、读懂 traceback |


---
## 1. 变量、数据类型、条件与循环

Junior DE 最常见的 Python 场景是处理一批记录。先把每条记录看成一个 `dict`，把多条记录看成一个 `list[dict]`。

要点：
- `None` 表示缺失值，判断时用 `is None`
- `if` 负责分支，`for` 负责遍历，`while` 只在循环次数不明确时使用
- f-string 用来生成清晰的日志或提示信息


In [6]:
import psycopg2
from psycopg2.extras import RealDictCursor
# 导入 psycopg2 库，用来让 Python 连接 PostgreSQL 数据库
conn = psycopg2.connect(
    host="localhost",
    database="postgres",
    user="postgres",
    password="postgres", 
    port=5432
)# 创建数据库连接对象 conn

cur = conn.cursor(cursor_factory=RealDictCursor)
# 创建 cursor 游标对象，用来执行 SQL 语句、获取查询结果
cur.execute("SELECT sale_id, salesperson, category, amount, sale_date FROM public.sales;")

rows = cur.fetchall()
for row in rows:
    print(row)
# 获取查询返回的所有结果，并保存到 rows 变量中
# rows 通常是一个 list，里面每一行是一个 tuple

cur.close()# 关闭 cursor，释放数据库操作资源
conn.close()# 关闭数据库连接

RealDictRow({'sale_id': 1, 'salesperson': 'Alice', 'category': 'Electronics', 'amount': Decimal('1200.00'), 'sale_date': datetime.date(2024, 1, 15)})
RealDictRow({'sale_id': 2, 'salesperson': 'Bob', 'category': 'Clothing', 'amount': Decimal('350.00'), 'sale_date': datetime.date(2024, 1, 16)})
RealDictRow({'sale_id': 3, 'salesperson': 'Alice', 'category': 'Electronics', 'amount': Decimal('800.00'), 'sale_date': datetime.date(2024, 1, 20)})
RealDictRow({'sale_id': 4, 'salesperson': 'Charlie', 'category': 'Electronics', 'amount': Decimal('950.00'), 'sale_date': datetime.date(2024, 1, 22)})
RealDictRow({'sale_id': 5, 'salesperson': 'Bob', 'category': 'Electronics', 'amount': Decimal('430.00'), 'sale_date': datetime.date(2024, 1, 25)})
RealDictRow({'sale_id': 6, 'salesperson': 'Alice', 'category': 'Clothing', 'amount': Decimal('200.00'), 'sale_date': datetime.date(2024, 2, 1)})
RealDictRow({'sale_id': 7, 'salesperson': 'Charlie', 'category': 'Clothing', 'amount': Decimal('150.00'), 'sale_da

In [4]:
conn.type()

AttributeError: 'psycopg2.extensions.connection' object has no attribute 'type'

In [ ]:
orders = [
    {"order_id": 1, "customer_id": "C001", "amount": 120.5, "status": "paid"},
    {"order_id": 2, "customer_id": "C002", "amount": 0, "status": "cancelled"},
    {"order_id": 3, "customer_id": "C001", "amount": 89.9, "status": "paid"},
]

paid_count = 0
paid_amount = 0.0

for order in orders:
    if order["status"] == "paid":
        paid_count += 1
        paid_amount += order["amount"]

print(f"paid orders={paid_count}, paid amount={paid_amount:.2f}")


---
## 2. 函数、参数、返回值、默认参数

函数是把脚本变成工程代码的第一步。一个好的 transform 函数通常有三个特点：输入明确、输出明确、不偷偷依赖全局变量。

练习时优先写纯函数：给它数据，返回新数据；不要在函数里随意读写文件。


In [ ]:
def filter_orders_by_status(records, status="paid"):
    """Return orders whose status matches the target status."""
    result = []
    for record in records:
        if record.get("status") == status:
            result.append(record)
    return result


def calculate_total_amount(records):
    total = 0.0
    for record in records:
        total += float(record.get("amount", 0))
    return total


paid_orders = filter_orders_by_status(orders)
print(paid_orders)
print(calculate_total_amount(paid_orders))


---
## 3. List / Dict / Set 常用操作

数据工程里经常要做去重、查找、按 key 聚合。`list` 适合保留顺序，`dict` 适合按 key 查找，`set` 适合去重和集合判断。


In [ ]:
customer_ids = [order["customer_id"] for order in orders]
unique_customer_ids = set(customer_ids)
print(unique_customer_ids)

amount_by_customer = {}
for order in orders:
    customer_id = order["customer_id"]
    amount_by_customer[customer_id] = amount_by_customer.get(customer_id, 0) + order["amount"]

print(amount_by_customer)


---
## 4. 推导式与 f-string

推导式适合写简单转换，不适合塞复杂业务逻辑。规则是：一眼能看懂就用推导式，看不懂就拆成普通 `for` 循环。


In [ ]:
paid_order_ids = [order["order_id"] for order in orders if order["status"] == "paid"]
status_by_order_id = {order["order_id"]: order["status"] for order in orders}

print(f"paid order ids: {paid_order_ids}")
print(f"status map: {status_by_order_id}")


---
## 5. pathlib 与 with open

`pathlib.Path` 比手写字符串路径更稳，能跨平台拼接路径。读写文件时用 `with open(...)`，这样文件会自动关闭。


In [ ]:
from pathlib import Path

work_dir = Path("tmp_python_basics")
work_dir.mkdir(exist_ok=True)

text_path = work_dir / "note.txt"

with text_path.open("w", encoding="utf-8") as f:
    f.write("daily order summary\n")
    f.write("rows=3\n")

with text_path.open("r", encoding="utf-8") as f:
    content = f.read()

print(content)


---
## 6. 读写 JSON 与 CSV

JSON 常见于 API 和半结构化数据，CSV 常见于导入导出和小规模交换。Junior 阶段要先会用标准库 `json` 和 `csv` 完成基本读写。


In [ ]:
import csv
import json

json_path = work_dir / "orders.json"
csv_path = work_dir / "orders.csv"

with json_path.open("w", encoding="utf-8") as f:
    json.dump(orders, f, ensure_ascii=False, indent=2)

with json_path.open("r", encoding="utf-8") as f:
    loaded_orders = json.load(f)

with csv_path.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["order_id", "customer_id", "amount", "status"])
    writer.writeheader()
    writer.writerows(loaded_orders)

with csv_path.open("r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    csv_orders = list(reader)

print(csv_orders[:2])


---
## 7. try / except 与 traceback

不要写裸 `except:`。捕获具体异常，才能知道你预期处理的是哪类失败。读 traceback 时从最后一段看起：异常类型、异常信息、出错文件和行号。


In [ ]:
def load_orders_json(path):
    try:
        with Path(path).open("r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError as exc:
        raise FileNotFoundError(f"Input file does not exist: {path}") from exc
    except json.JSONDecodeError as exc:
        raise ValueError(f"Input file is not valid JSON: {path}") from exc


print(load_orders_json(json_path))


---
## 阶段验收

完成后你应该能独立做到：

1. 用 `list[dict]` 表达一批业务记录
2. 用函数过滤、转换、汇总记录，并返回清晰结果
3. 使用 `pathlib` 和 `with open` 读写文本文件
4. 用标准库读写 JSON 和 CSV
5. 捕获 `FileNotFoundError`、`JSONDecodeError` 等具体异常
6. 根据 traceback 定位到出错文件、行号和异常类型

练习项目：读取订单 JSON，过滤 `status == "paid"` 的订单，按 `customer_id` 汇总金额，输出 summary JSON 和 summary CSV。
